In [ ]:
# Table discovery for bulk onboarding (run by the lakefed_ingest_discover job; usually started from
# the operations app). Lists every user table of each requested source database with recommended
# control-row settings (discovery.py) and writes them to Lakebase for review. Catalog metadata
# only: no table data is read. Runs on the same compute and network path as ingestion.
#
# spec (JSON): {"src_type": "sqlserver", "src_host": "...", "src_port": 1433 | null,
#               "databases": ["db1", "db2"], "secret_key": "..." | "secret_key_template": "creds-{database}",
#               "requested_by": "..."}
import json, os, sys

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import ct_engine, discovery
from state_store import StateStore

def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)

discovery_id = widget("discovery_id")
spec = json.loads(widget("spec", "{}"))
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
os.environ["LAKEFED_SQL_SECRET_SCOPE"] = widget("secret_scope", "lakefed_ingest")
os.environ["LAKEFED_SQL_TRUST_SERVER_CERT"] = widget("trust_server_certificate", "true")
databases = spec.get("databases") or []
assert discovery_id and databases and spec.get("src_host"), "discovery_id, spec.src_host and spec.databases are required"
assert spec.get("secret_key") or spec.get("secret_key_template"), "spec needs secret_key or secret_key_template"
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

TABLE_COLS = ["discovery_id", "src_database", "src_schema", "src_table", "row_count", "size_mb", "ct_enabled",
              "primary_key", "select_list", "excluded_columns", "rowversion_column", "identity_column",
              "load_type", "cursor_column", "partition_candidate", "notes"]

def start(db):
    with store.connection.cursor() as cur:
        cur.execute(
            "insert into lakefed_ingest.discovery (discovery_id, src_type, src_host, src_port, src_database, secret_key, "
            "requested_by) values (%s, %s, %s, %s, %s, %s, %s) on conflict (discovery_id, src_database) do update set "
            "status = 'running', error = null, started_at = now(), finished_at = null",
            (discovery_id, spec.get("src_type") or "sqlserver", spec["src_host"], spec.get("src_port"), db,
             discovery.secret_key_for(spec, db), spec.get("requested_by")))

def finish(db, status, tables=None, error=None):
    with store.connection.cursor() as cur:
        cur.execute("update lakefed_ingest.discovery set status = %s, tables = %s, error = %s, finished_at = now() "
                    "where discovery_id = %s and src_database = %s", (status, tables, error, discovery_id, db))

def one(db):
    start(db)
    cfg = {"id": 0, "src_type": spec.get("src_type") or "sqlserver", "src_host": spec["src_host"],
           "src_port": spec.get("src_port"), "src_database": db, "secret_key": discovery.secret_key_for(spec, db)}
    try:
        found = discovery.discover_database(cfg)
    except Exception as e:
        finish(db, "failed", error=str(e)[:2000])
        return db, None, str(e)[:300]
    rows = [tuple({**t, "discovery_id": discovery_id, "src_database": db}.get(c) for c in TABLE_COLS) for t in found]
    with store.connection.cursor() as cur:
        cur.execute("delete from lakefed_ingest.discovered_table where discovery_id = %s and src_database = %s",
                    (discovery_id, db))
        cur.executemany(f"insert into lakefed_ingest.discovered_table ({', '.join(TABLE_COLS)}) "
                        f"values ({', '.join(['%s'] * len(TABLE_COLS))})", rows)
    finish(db, "ok", tables=len(rows))
    return db, len(rows), None

with ThreadPoolExecutor(max_workers=min(8, len(databases))) as ex:
    results = list(ex.map(one, databases))
for db, n, err in results:
    print(f"{db}: {n} tables" if err is None else f"{db}: FAILED {err}")
failed = [db for db, _, err in results if err]
store.close()
summary = {"discovery_id": discovery_id, "databases": len(databases), "failed": failed,
           "tables": sum(n or 0 for _, n, _ in results)}
dbutils.notebook.exit(json.dumps(summary))